<a href="https://colab.research.google.com/github/tendolkarurja/LogisticsOperations/blob/main/LogisticsSupply_initial_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import pandas as pd
import numpy as np

### Performed the following tests with Logistics and supply chain dataset to assess its usability.

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/dynamic_supply_chain_logistics_dataset.csv')

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df.isna().sum()

In [ ]:
df.info()

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['year'] = df.timestamp.dt.year
df['month'] = df.timestamp.dt.month
df['month_name'] = df['timestamp'].dt.month_name()
df['day'] = df['timestamp'].dt.day
df['day_of_week'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour
df['quarter'] = df['timestamp'].dt.quarter
df['week'] = df['timestamp'].dt.isocalendar().week
df['is_weekend'] = df['day_of_week'].isin(['Saturday', 'Sunday'])

In [ ]:
df

In [ ]:
df.timestamp.unique()

In [ ]:
df.timestamp.dt.date.unique()

In [ ]:
# Find missing timestamps
missing_timestamps = all_dates.difference(df['timestamp'])

print(f"Missing timestamps: {len(missing_timestamps)}")
print(missing_timestamps)

In [ ]:
print(df['timestamp'].duplicated().sum())

In [ ]:
df['risk_classification'].value_counts()

In [ ]:
df['cargo_condition_status']

In [ ]:
df['handling_equipment_availability']

In [ ]:
df['order_fulfillment_status']

In [ ]:
df['historical_demand']

## CANNOT BE USED, too synthetic with values of columns not very much justifiable, especially the binary ones which are represented as scale, causing error in judgement and interpretation

### Going ahead with Global Supply Chain Risk & Logistics (2024-2026), which is synthetic but values can be interpreted and justified

#

In [ ]:
df = pd.read_csv('/content/global_supply_chain_risk_2026.csv')

In [ ]:
df

In [ ]:


df.isna().sum()

In [ ]:
df.info()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
print(min(df.Date))
print(max(df.Date))

In [ ]:
all_dates = pd.date_range(
    start = '2024-01-01 00:00:00',
    end = '2025-12-31 00:00:00',
    freq = 'd'
)

In [ ]:

missing = all_dates.difference(df['Date'])
print(missing)

In [ ]:
df.Date.value_counts()

In [ ]:
df['Origin_Port'].value_counts()

In [ ]:
df['Destination_Port'].value_counts()

In [ ]:
df['Carrier_Reliability_Score']

In [ ]:
print(min(df['Carrier_Reliability_Score']), max(df['Carrier_Reliability_Score']))

In [ ]:

df['Transport_Mode'].value_counts()

In [ ]:
df['Product_Category'].value_counts()

In [ ]:
print(min(df['Fuel_Price_Index']), max(df['Fuel_Price_Index']))

In [ ]:
df['Geopolitical_Risk_Score'].value_counts()

In [ ]:
def categ_trial(low_thresh, med_thresh):
    low = 0
    med = 0
    high = 0

    for score in df['Geopolitical_Risk_Score']:
        if score > med_thresh:
            high += 1
        elif score > low_thresh:
            med += 1
        else:
            low += 1
    return low, med, high

In [ ]:
thresholds = [(3,7), (3,7.5), (3, 8), (3.5, 7), (3.5, 7.5), (3.5, 8), (4, 7), (4, 7.5), (4, 8)]

for low, med in thresholds:
    print(f'Thresh: {low}, {med}, Dist: ', categ_trial(low, med))

### Given the above trials, geopolitical risk would be binned into three groups =>
#### Low risk - less than 3.5
#### Moderate risk - less than 7
#### High risk - 7 and above

In [ ]:
def categ_assign(score, low_thresh=3.5, med_thresh=7):
    if score >= med_thresh:
        return "High"
    elif score >= low_thresh:
        return "Moderate"
    else:
        return "Low"

In [ ]:
df['geopolitical_risk_class'] = df['Geopolitical_Risk_Score'].apply(categ_assign)

In [ ]:
df

In [ ]:
df['geopolitical_risk_class'].value_counts()

In [ ]:
df['Lead_Time_Days']

##### hour min sec condensed into decumal by div by 24

In [ ]:
df['Weather_Condition'].value_counts()

In [ ]:
df['Disruption_Occurred'].value_counts()

In [ ]:
print(min(df['Weight_MT']), max(df['Weight_MT']))

In [ ]:

print(min(df['Distance_km']), max(df['Distance_km']))

In [ ]:

df.duplicated().sum()

In [ ]:
df['Fuel_Price_Index'].describe()

### The dataset is good, now performing some feature engineering additionally for further analysis.

In [ ]:
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.month_name()
df['Quarter'] = df['Date'].dt.quarter
df['Day'] = df['Date'].dt.day
df['Weekday'] = df['Date'].dt.day_name()
df['Week'] = df['Date'].dt.isocalendar().week

In [ ]:
df

In [ ]:
df['Weekend'] = df['Weekday'].apply(lambda x: 1 if x in ['Saturday', 'Sunday'] else 0)
df['Weekend'].value_counts()

In [ ]:
df["Trade_Corridor"] = df.apply(
    lambda x: " - ".join(sorted([x["Origin_Port"], x["Destination_Port"]])),
    axis=1
)

In [ ]:
df

In [ ]:
df['Trade_Corridor'].value_counts()

#### Some basic visualizations:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
distrib = (
    df.groupby(["Year", "Quarter"])
      .size()
      .reset_index(name="Shipment_Count")
)

distrib["Period"] = (
    distrib["Year"].astype(str)
    + "-Q"
    + distrib["Quarter"].astype(str)
)

plt.figure(figsize=(10,5))
plt.plot(distrib["Period"], distrib["Shipment_Count"], marker="o")
plt.xlabel("Quarter")
plt.ylabel("Shipment Count")
plt.title("Quarterly Shipment Distribution")
plt.grid(True)
plt.show()

In [ ]:
num_cols = [
    "Distance_km",
    "Weight_MT",
    "Fuel_Price_Index",
    "Lead_Time_Days",
    "Geopolitical_Risk_Score",
    "Carrier_Reliability_Score"
]

corr = df[num_cols].corr()

In [ ]:
sns.heatmap(corr, annot=True)

In [ ]:
weather_disruption = (
    df.groupby("Weather_Condition")["Disruption_Occurred"]
      .mean()
      .reset_index()
)

sns.barplot(
    data=weather_disruption,
    x="Weather_Condition",
    y="Disruption_Occurred"
)

plt.ylabel("Disruption Rate")
plt.xlabel("Weather Condition")
plt.title("Disruption Rate by Weather Condition")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.boxplot(
    data=df,
    x="Transport_Mode",
    y="Lead_Time_Days"
)

plt.title("Lead Time by Transport Mode")
plt.show()

In [ ]:
df.info()

In [ ]:
sns.scatterplot(df, x = "Distance_km", y = "Lead_Time_Days", hue = "Transport_Mode")

In [ ]:
sns.histplot(
    data=df,
    x="Lead_Time_Days",
    bins=30,
    kde=True
)

In [ ]:
sns.countplot(
    data=df,
    x="Transport_Mode"
)

In [ ]:
sns.barplot(
    data=df,
    x="geopolitical_risk_class",
    y="Lead_Time_Days"
)